# Load gold.sales_fact

Builds the sales fact table from `silver.sales` by resolving surrogate keys against all
silver dimension tables.

Write strategy: INSERT OVERWRITE — atomically replaces all rows on every run.
Silver is itself an INSERT OVERWRITE table (full rebuild each run), so gold must
be fully rebuilt to stay consistent. MERGE cannot handle deletes without a second
DELETE pass; INSERT OVERWRITE is simpler and provides atomic reader visibility.

`sales_fact_id` (IDENTITY) reassigns on every run — downstream joins use
date_id, product_id, etc., not the fact surrogate key.

Unresolved FK lookups receive -1 (COALESCE guarantee — never NULL). Counts are
printed as warnings; they do not fail the load.

Starts with `%run "../../libs/notebook_init"` — see
`.claude/project/helpers.md` for what `notebook_init` injects.


In [0]:
%run "../../libs/notebook_init"

In [0]:
from datetime import datetime, timezone
import time

# transform_detail_log_insert isn't in notebook_init's central import yet —
# pull it in here so this notebook can log per-transform audit rows.
from pipeline_logging import transform_detail_log_insert

TARGET_TABLE      = f"{GOLD}.sales_fact"
SOURCE_TABLE      = f"{SILVER}.sales"
DIM_DATE          = f"{SILVER}.dim_date"
DIM_CURRENCY      = f"{SILVER}.dim_currency"
DIM_STORE         = f"{SILVER}.dim_store"
DIM_TERRITORY     = f"{SILVER}.dim_territory"
DIM_PRODUCT       = f"{SILVER}.dim_product"
DIM_EXCHANGE_RATE = f"{SILVER}.dim_exchange_rate"
DIM_REGION        = f"{SILVER}.dim_region"


In [0]:
nb = Utils.get_notebook_context(dbutils)
notebook_folder = nb['notebook_folder']
notebook_name   = nb['notebook_name']

step_log_id       = str(uuid.uuid4())
pipeline_run_id   = PIPELINE_RUN_ID
step_sequence     = 1
layer             = "gold"
target_table      = TARGET_TABLE
status            = STATUS_RUNNING
started_timestamp = datetime.now(timezone.utc)
rows_read         = 0
rows_written      = 0
error_message     = None

pipeline_step_log_upsert(
    spark, step_log_id, pipeline_run_id, step_sequence,
    notebook_folder, notebook_name, status, started_timestamp,
    layer, target_table
)

In [0]:
# ── Build vw_gold_sales_staging ───────────────────────────────────────────────
#
# Full extraction with all dim joins, QUALIFY deduplication, row_hash, and
# _warn_* boolean flags for unresolved FK detection.
#
# Join corrections applied (vs. prototype):
#   dim_exchange_rate: AND der.EffectiveDate = dd.LastDayOfMonth (date-specific rate)
#   dim_region:        AND dr.Province = dp.Province (prevents fan-out on duplicate
#                      region names across provinces)
# ─────────────────────────────────────────────────────────────────────────────

try:
    spark.sql(f"""
        CREATE OR REPLACE TEMPORARY VIEW vw_gold_sales_staging AS
        WITH sales_staging AS (
            SELECT
                COALESCE(dd.DateId,      -1) AS date_id,
                COALESCE(dp.ProductId,   -1) AS product_id,
                COALESCE(ds.StoreId,     -1) AS store_id,
                COALESCE(dt.TerritoryId, -1) AS territory_id,
                COALESCE(dr.RegionId,    -1) AS region_id,
                COALESCE(dc.CurrencyId,  -1) AS currency_id,
                s.product_no,
                s.sales_month,
                s.quantity,
                CAST(s.list_price AS DECIMAL(10,2))                          AS list_price_local,
                CAST(s.quantity * s.list_price AS DECIMAL(10,2))             AS total_sales,
                der.ExchangeRate                                              AS exchange_rate_applied,
                TRY_CAST(
                    CASE WHEN der.FromCurrency IS NULL
                         THEN s.list_price
                         ELSE s.list_price * der.ExchangeRate
                    END AS DECIMAL(10,2)
                )                                                            AS list_price_converted,
                TRY_CAST(
                    CASE WHEN der.FromCurrency IS NULL
                         THEN s.quantity * s.list_price
                         ELSE s.quantity * s.list_price * der.ExchangeRate
                    END AS DECIMAL(10,2)
                )                                                            AS total_sales_converted,
                (dd.DateId      IS NULL) AS _warn_date,
                (dp.ProductId   IS NULL) AS _warn_product,
                (ds.StoreId     IS NULL) AS _warn_store,
                (dt.TerritoryId IS NULL) AS _warn_territory,
                (dr.RegionId    IS NULL) AS _warn_region,
                (dc.CurrencyId  IS NULL) AS _warn_currency
            FROM {SOURCE_TABLE} s
            LEFT JOIN {DIM_DATE}          dd   ON  dd.YearMonth     = s.sales_month
            LEFT JOIN {DIM_CURRENCY}      dc   ON  dc.CurrencyCode  = s.sales_currency
            LEFT JOIN {DIM_STORE}         ds   ON  ds.StoreName     = s.online_retailer
            LEFT JOIN {DIM_TERRITORY}     dt   ON  dt.TerritoryCode = s.sales_territory
            LEFT JOIN {DIM_PRODUCT}       dp   ON  dp.ProductNo     = s.product_no
                                               AND dp.IsRowCurrent  = TRUE
            LEFT JOIN {DIM_EXCHANGE_RATE} der  ON  der.FromCurrency = s.sales_currency
                                               AND der.EffectiveDate = dd.LastDayOfMonth
            LEFT JOIN {DIM_REGION}        dr   ON  dr.RegionName    = dp.Region
                                               AND dr.Province      = dp.Province
            QUALIFY ROW_NUMBER() OVER (
                PARTITION BY s.product_no, s.sales_month
                ORDER BY s.source_inserted_ts DESC
            ) = 1
        )
        SELECT
            date_id,
            product_id,
            store_id,
            territory_id,
            region_id,
            currency_id,
            product_no,
            sales_month,
            quantity,
            list_price_local,
            total_sales,
            exchange_rate_applied,
            list_price_converted,
            total_sales_converted,
            MD5(CONCAT_WS('|',
                CAST(date_id                         AS STRING),
                CAST(product_id                      AS STRING),
                CAST(store_id                        AS STRING),
                CAST(territory_id                    AS STRING),
                CAST(region_id                       AS STRING),
                CAST(currency_id                     AS STRING),
                product_no,
                sales_month,
                CAST(quantity                        AS STRING),
                CAST(list_price_local                AS STRING),
                CAST(total_sales                     AS STRING),
                COALESCE(CAST(exchange_rate_applied  AS STRING), ''),
                COALESCE(CAST(list_price_converted   AS STRING), ''),
                COALESCE(CAST(total_sales_converted  AS STRING), '')
            ))                           AS row_hash,
            CURRENT_TIMESTAMP()          AS inserted_ts,
            CURRENT_TIMESTAMP()          AS updated_ts,
            _warn_date,
            _warn_product,
            _warn_store,
            _warn_territory,
            _warn_region,
            _warn_currency
        FROM sales_staging
    """)

    print("Staging view created: vw_gold_sales_staging")

except dbutils.NotebookExit:
    raise

except Exception as e:
    err = Utils.capture_exception(e)
    error_message = (
        f"{err['error_type']}: {err['error_message']}\n\n"
        f"{err['error_traceback']}"
    )
    ended_timestamp = datetime.now(timezone.utc)
    status = STATUS_FAILED
    pipeline_step_log_upsert(
        spark, step_log_id, pipeline_run_id, step_sequence,
        notebook_folder, notebook_name, status, started_timestamp,
        layer, target_table, rows_read, rows_written, ended_timestamp, error_message
    )
    raise

In [0]:
# ── INSERT OVERWRITE — atomic full replace ────────────────────────────────────
#
# Collect warn counts from the staging view before the write so warnings are
# available even if the assertion fails. sales_fact_id excluded from the column
# list — Delta assigns it via GENERATED ALWAYS AS IDENTITY. _warn_* columns
# excluded — used only for load-time diagnostics, not stored in gold.
# ─────────────────────────────────────────────────────────────────────────────

# Per-transform variables for transform_detail_log. Defined OUTSIDE the try
# block so the except handler can log a failed transform row even if the
# INSERT OVERWRITE never started.
transform_source_table = "silver.sales + dim_date + dim_product + dim_store + dim_territory + dim_region + dim_currency + dim_exchange_rate"
transform_target_table = TARGET_TABLE
transform_started      = datetime.now(timezone.utc)
unresolved_total       = 0

try:
    rows_read = spark.sql("SELECT COUNT(*) FROM vw_gold_sales_staging").collect()[0][0]

    warn_counts = spark.sql("""
        SELECT
            SUM(CAST(_warn_date      AS INT)) AS unresolved_date,
            SUM(CAST(_warn_product   AS INT)) AS unresolved_product,
            SUM(CAST(_warn_store     AS INT)) AS unresolved_store,
            SUM(CAST(_warn_territory AS INT)) AS unresolved_territory,
            SUM(CAST(_warn_region    AS INT)) AS unresolved_region,
            SUM(CAST(_warn_currency  AS INT)) AS unresolved_currency
        FROM vw_gold_sales_staging
    """).collect()[0]

    # Sum of unresolved-FK warnings → rows_rejected on transform_detail_log.
    # These rows aren't quarantined; they land in gold with FK = -1. We use
    # rows_rejected as the closest-fit existing column.
    unresolved_total = sum((c or 0) for c in warn_counts.asDict().values())

    spark.sql(f"""
        INSERT OVERWRITE {TARGET_TABLE}
            (date_id, product_id, store_id, territory_id, region_id, currency_id,
             product_no, sales_month, quantity, list_price_local, total_sales,
             exchange_rate_applied, list_price_converted, total_sales_converted,
             row_hash, inserted_ts, updated_ts)
        SELECT
            date_id, product_id, store_id, territory_id, region_id, currency_id,
            product_no, sales_month, quantity, list_price_local, total_sales,
            exchange_rate_applied, list_price_converted, total_sales_converted,
            row_hash, inserted_ts, updated_ts
        FROM vw_gold_sales_staging
    """)

    rows_written = spark.table(TARGET_TABLE).count()

    if rows_written != rows_read:
        raise AssertionError(
            f"[{TARGET_TABLE}] Row count mismatch: "
            f"staging view had {rows_read:,} rows, "
            f"table has {rows_written:,} rows after INSERT OVERWRITE."
        )

    for dim, count in warn_counts.asDict().items():
        if count and count > 0:
            print(f"[WARNING] [{TARGET_TABLE}] {count:,} rows have unresolved {dim} (FK = -1)")

    print(f"[{TARGET_TABLE}] {rows_written:,} rows written.")

    ended_timestamp = datetime.now(timezone.utc)
    status = STATUS_SUCCEEDED

    transform_detail_log_insert(
        spark,
        pipeline_run_id          = PIPELINE_RUN_ID,
        step_log_id              = step_log_id,
        source_table             = transform_source_table,
        target_table             = transform_target_table,
        status                   = status,
        started_timestamp        = transform_started,
        ended_timestamp          = ended_timestamp,
        rows_read                = rows_read,
        rows_written             = rows_written,
        rows_inserted            = rows_written,
        rows_rejected            = unresolved_total,
        validation_rules_applied = '["FK resolution: dim_date, dim_product, dim_store, dim_territory, dim_region, dim_currency, dim_exchange_rate"]',
    )

    pipeline_step_log_upsert(
        spark, step_log_id, pipeline_run_id, step_sequence,
        notebook_folder, notebook_name, status, started_timestamp,
        layer, target_table, rows_read, rows_written, ended_timestamp, error_message
    )

except dbutils.NotebookExit:
    raise

except Exception as e:
    err = Utils.capture_exception(e)
    error_message = (
        f"{err['error_type']}: {err['error_message']}\n\n"
        f"{err['error_traceback']}"
    )
    ended_timestamp = datetime.now(timezone.utc)
    status = STATUS_FAILED

    transform_detail_log_insert(
        spark,
        pipeline_run_id   = PIPELINE_RUN_ID,
        step_log_id       = step_log_id,
        source_table      = transform_source_table,
        target_table      = transform_target_table,
        status            = status,
        started_timestamp = transform_started,
        ended_timestamp   = ended_timestamp,
        rows_read         = rows_read,
        rows_rejected     = unresolved_total,
        error_message     = error_message,
    )

    pipeline_step_log_upsert(
        spark, step_log_id, pipeline_run_id, step_sequence,
        notebook_folder, notebook_name, status, started_timestamp,
        layer, target_table, rows_read, rows_written, ended_timestamp, error_message
    )
    raise


In [0]:
%skip
%sql
-- Spot-check: FK resolution rate by dimension
SELECT
    COUNT(*)                                              AS total_rows,
    SUM(CASE WHEN date_id      = -1 THEN 1 ELSE 0 END)  AS unresolved_date,
    SUM(CASE WHEN product_id   = -1 THEN 1 ELSE 0 END)  AS unresolved_product,
    SUM(CASE WHEN store_id     = -1 THEN 1 ELSE 0 END)  AS unresolved_store,
    SUM(CASE WHEN territory_id = -1 THEN 1 ELSE 0 END)  AS unresolved_territory,
    SUM(CASE WHEN region_id    = -1 THEN 1 ELSE 0 END)  AS unresolved_region,
    SUM(CASE WHEN currency_id  = -1 THEN 1 ELSE 0 END)  AS unresolved_currency
FROM vinoworld.gold.sales_fact

In [0]:
%skip
%sql
-- Sample rows with currency conversion
SELECT
    product_no, sales_month, quantity,
    list_price_local, total_sales,
    exchange_rate_applied,
    list_price_converted, total_sales_converted
FROM vinoworld.gold.sales_fact
WHERE exchange_rate_applied IS NOT NULL
LIMIT 20

In [0]:
%skip
%sql
SELECT COUNT(*) FROM vinoworld.gold.sales_fact